In [1]:
import numpy as np
import pandas as pd 
import seaborn as sns 
import anndata as ad 
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.ticker import ScalarFormatter

# load splice anndata in human 
HUMAN_SPLICE_INPUT = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/HUMAN_SPLICING_FOUNDATION/MODEL_INPUT/062025/aligned_splicing_data_20250627_163745.h5ad"

# load splice anndata in mouse
MOUSE_SPLICE_INPUT = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/MOUSE_SPLICING_FOUNDATION/MODEL_INPUT/062025/aligned_splicing_data_20250625_182138.h5ad"


In [2]:
human = ad.read_h5ad(HUMAN_SPLICE_INPUT)
print(f"Finished Reading Human dataset...")
mouse = ad.read_h5ad(MOUSE_SPLICE_INPUT)
print(f"Finished Reading Mouse dataset...")

/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Finished Reading Human dataset...
Finished Reading Mouse dataset...


In [3]:
# of junctions detected per cell 
# of ATSEs detected per cell 
# junction counts vs gene counts...?

In [8]:
human.obs[["dataset"]].value_counts()

dataset       
allen_brain       46546
tabula_sapiens    34358
Name: count, dtype: int64

In [9]:
mouse.obs[["dataset", "age"]].value_counts()

dataset  age
AB       2m     50787
TMS      3m     40103
         18m    28831
         24m    28274
Name: count, dtype: int64

In [11]:
40103+28831+28274+50787

147995

In [12]:
46546+34358

80904

In [13]:
147995+80904

228899

In [ ]:
# 1. summarize number of junctions detected in a cell 
# Ensure correct matrix
cell_by_junction = human.layers["cell_by_junction_matrix"]
if not sp.issparse(cell_by_junction):
    cell_by_junction = sp.csr_matrix(cell_by_junction)

# Compute per-cell read counts
human.obs["reads_per_cell"] = np.ravel(cell_by_junction.sum(axis=1))

# Compute per-cell non-zero junction count
human.obs["junctions_detected_per_cell"] = np.ravel((cell_by_junction > 0).sum(axis=1))

# Compute per-junction read counts
human.var["reads_per_junction"] = np.ravel(cell_by_junction.sum(axis=0))

# ---- Summary Statistics ----
print("Reads per cell:\n", human.obs["reads_per_cell"].describe())
print("\nJunctions with reads > 0 per cell:\n", human.obs["junctions_detected_per_cell"].describe())
print("\nReads per junction:\n", human.var["reads_per_junction"].describe())

In [ ]:
# === Plot 1: Reads per Cell by Dataset (with scientific notation) ===
plt.figure(figsize=(5, 5))
sns.histplot(data=human.obs, x="reads_per_cell", hue="dataset", bins=100,
             multiple="layer", element="step")
plt.title("Read Counts per Cell by Dataset", fontsize=14)
plt.xlabel("Reads per Cell", fontsize=12)
plt.ylabel("Cell Count", fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.xlim(0, np.percentile(human.obs["reads_per_cell"], 99))

# Use scientific notation on x-axis
ax = plt.gca()
ax.xaxis.set_major_formatter(ScalarFormatter(useMathText=True))
ax.ticklabel_format(style="sci", axis="x", scilimits=(0, 0))

# Add median lines + labels
ylim = plt.ylim()
line_spacing = (ylim[1] * 0.9) / (len(human.obs["dataset"].unique()) + 1)
for i, ds in enumerate(sorted(human.obs["dataset"].unique())):
    median_val = human.obs.loc[human.obs["dataset"] == ds, "reads_per_cell"].median()
    plt.axvline(median_val, linestyle="--", linewidth=1, alpha=0.7)
    plt.text(plt.xlim()[1] * 0.98, ylim[1] - i * line_spacing,
             f"{ds} median = {int(median_val):.1e}",
             ha="right", va="top", fontsize=9)

plt.tight_layout()
plt.savefig("reads_per_cell_by_dataset.pdf")

# === Plot 2: Junctions Detected per Cell ===
plt.figure(figsize=(5, 5))
sns.histplot(data=human.obs, x="junctions_detected_per_cell", hue="dataset", bins=100,
             multiple="layer", element="step")
plt.title("Junctions with Reads > 0 per Cell", fontsize=14)
plt.xlabel("Detected Junctions per Cell", fontsize=12)
plt.ylabel("Cell Count", fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.xlim(0, np.percentile(human.obs["junctions_detected_per_cell"], 99))

ylim = plt.ylim()
line_spacing = (ylim[1] * 0.9) / (len(human.obs["dataset"].unique()) + 1)
for i, ds in enumerate(sorted(human.obs["dataset"].unique())):
    median_val = human.obs.loc[human.obs["dataset"] == ds, "junctions_detected_per_cell"].median()
    plt.axvline(median_val, linestyle="--", linewidth=1, alpha=0.7)
    plt.text(plt.xlim()[1] * 0.98, ylim[1] - i * line_spacing,
             f"{ds} median = {int(median_val)}",
             ha="right", va="top", fontsize=9)

plt.tight_layout()
plt.savefig("junctions_per_cell_by_dataset.pdf")



In [ ]:
# === Plot 3: Reads per Junction ===
plt.figure(figsize=(5, 5))
sns.histplot(human.var["reads_per_junction"])
plt.title("Read Counts per Junction", fontsize=14)
plt.xlabel("Reads per Junction", fontsize=12)
plt.ylabel("Junction Count", fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.xlim(0, np.percentile(human.var["reads_per_junction"], 99))

median_val = human.var["reads_per_junction"].median()
plt.axvline(median_val, linestyle="--", linewidth=1, alpha=0.7)
plt.text(plt.xlim()[1] * 0.98, plt.ylim()[1]*0.9,
         f"median = {int(median_val)}", ha="right", va="top", fontsize=9)

plt.tight_layout()
plt.savefig("reads_per_junction.pdf")